# License Plate Detection and Recognition
This notebook demonstrates the full pipeline for license plate detection using YOLOv5 and recognition using EasyOCR.
Dataset Source: [Roboflow - License Plate Recognition](https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/4)

In [ ]:
# Install dependencies
!pip install ultralytics easyocr roboflow opencv-python-headless matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set dataset path
dataset_path = '/content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e'
!find $dataset_path -name data.yaml

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace().project("license-plate-recognition-rxg4e")
dataset = project.version(4).download("yolov5", location="/content/drive/My Drive/747/Project")


In [ ]:
import os
import random
import shutil

src_images = "/content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e/train/images"
src_labels = "/content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e/train/labels"
dst_images = "/content/sample_dataset/train/images"
dst_labels = "/content/sample_dataset/train/labels"

os.makedirs(dst_images, exist_ok=True)
os.makedirs(dst_labels, exist_ok=True)

# Only sample images that have corresponding label files
valid_images = []
for f in os.listdir(src_images):
    if f.endswith(".jpg") or f.endswith(".png"):
        label = f.rsplit(".", 1)[0] + ".txt"
        if os.path.exists(os.path.join(src_labels, label)):
            valid_images.append(f)

# Sample 1000 images that have labels
sampled_images = random.sample(valid_images, 1000)

for img in sampled_images:
    shutil.copy(os.path.join(src_images, img), dst_images)
    label_file = img.rsplit(".", 1)[0] + ".txt"
    shutil.copy(os.path.join(src_labels, label_file), dst_labels)

print(f"✅ Copied {len(sampled_images)} training samples with labels.")

In [ ]:
# Extract test images that are NOT in training set
test_images_dir = "/content/sample_dataset/test_images"
os.makedirs(test_images_dir, exist_ok=True)

# All valid images with labels
all_valid_images = set(valid_images)
used_in_train = set(sampled_images)
remaining_images = list(all_valid_images - used_in_train)

# Sample 100 test images
test_samples = random.sample(remaining_images, 100)

# Copy test images only (no need for labels during detection)
for img in test_samples:
    shutil.copy(os.path.join(src_images, img), test_images_dir)

print(f"✅ Copied {len(test_samples)} test images to: {test_images_dir}")


## Step 1: Train YOLOv5 on License Plate Detection

In [ ]:
# Clone YOLOv5 and install dependencies
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
yaml_text = """path: /content/sample_dataset

train: train/images
val: /content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e/valid/images
test: /content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e/test/images

nc: 1
names: ['License_Plate']
"""

with open("/content/sample_dataset/data.yaml", "w") as f:
    f.write(yaml_text)

print("✅ data.yaml created.")


In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 20 \
  --data /content/sample_dataset/data.yaml \
  --weights yolov5s.pt \
  --name lp_detector_small

In [ ]:
!python detect.py \
  --weights runs/train/lp_detector_small3/weights/best.pt \
  --img 640 \
  --conf 0.4 \
  --source /content/sample_dataset/test_images \
  --save-crop \
  --save-txt \
  --project runs/detect \
  --name test_lp \
  --exist-ok

In [ ]:
!apt install tesseract-ocr -y
!pip install pytesseract


In [ ]:
import pytesseract
from PIL import Image
import os

# Path to cropped license plates
cropped_dir = '/content/yolov5/runs/detect/test_lp/crops/License_Plate'

# Perform OCR on each cropped image
for filename in os.listdir(cropped_dir):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        path = os.path.join(cropped_dir, filename)
        img = Image.open(path)
        text = pytesseract.image_to_string(img, config='--psm 7')  # Treat as a single text line
        print(f'{filename}: {text.strip()}')


In [ ]:
from matplotlib import pyplot as plt
import cv2
import torch

model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/lp_detector_small/weights/best.pt')
img_path = '/content/sample_dataset/train/images'
img_file = os.listdir(img_path)[0]
img_full = os.path.join(img_path, img_file)

results = model(img_full)
results.print()
results.show()

# Save results with bounding boxes
results.save()

In [ ]:
import easyocr

reader = easyocr.Reader(['en'])

# Get results in array form
boxes = results.pandas().xyxy[0]
img = cv2.imread(img_full)

for i, row in boxes.iterrows():
    x1, y1, x2, y2 = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
    plate_crop = img[y1:y2, x1:x2]
    result = reader.readtext(plate_crop)
    print(f"OCR Result for box {i}: {result}")
    if result:
        text = result[0][1]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(img, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

# Show OCR annotated image
plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Detected Plate with OCR")
plt.show()


In [ ]:
from IPython.display import Image
Image(filename='runs/detect/exp/' + img_file)


In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 50 \
  --data /content/drive/MyDrive/747/Project/license-plate-recognition-rxg4e/data.yaml \
  --weights yolov5s.pt \
  --name lp_detector

In [ ]:
!apt install tesseract-ocr -y
!pip install pytesseract


In [ ]:
import pytesseract
from PIL import Image
import os

crop_dir = "runs/detect/lp_test_output"

ocr_results = []
for fname in os.listdir(crop_dir):
    if fname.endswith(".jpg") or fname.endswith(".png"):
        img_path = os.path.join(crop_dir, fname)
        img = Image.open(img_path)
        text = pytesseract.image_to_string(img, config='--psm 7')  # PSM 7: single text line
        ocr_results.append((fname, text.strip()))

# Print the OCR results
for fname, text in ocr_results:
    print(f"{fname}: {text}")


## Step 2: License Plate OCR using EasyOCR

In [ ]:
import easyocr, cv2, matplotlib.pyplot as plt
reader = easyocr.Reader(['en'])

# Replace with actual image and bounding box coordinates
img_path = '../{dataset.location}/test/images/sample.jpg'
img = cv2.imread(img_path)
plate_crop = img[y1:y2, x1:x2]  # Replace with YOLO output
result = reader.readtext(plate_crop)
print(result)